In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from tqdm import tqdm
import optuna
import wandb


In [2]:
# Basic settings
# ==========================================
IMAGE_DIR = r"D:\COURSE_DATA\Intro_Deep_Learning\project\data\Post_Impressionism"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS_SEARCH = 5
SEARCH_TIMEOUT_SEC = 1800  # 30 minutes limit

print(f"✅ Device detected: {DEVICE}")

✅ Device detected: cuda


In [3]:
# Data preparations
# ==========================================
def prepare_data_indices(dir_path):
    if not os.path.exists(dir_path):
        print(f"❌ Error: Folder not found at {dir_path}")
        exit()

    all_files = [f for f in os.listdir(dir_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    labels = [1 if f.lower().startswith("vincent-van-gogh") else 0 for f in all_files]

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        all_files, labels, test_size=0.10, random_state=42, stratify=labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.1111, random_state=42, stratify=y_train_val
    )
    return X_train, y_train, X_val, y_val


print("⏳ Scanning files...")
X_train, y_train, X_val, y_val = prepare_data_indices(IMAGE_DIR)
print(f"✅ Data loaded! Train: {len(X_train)} | Val: {len(X_val)}")

⏳ Scanning files...
✅ Data loaded! Train: 5160 | Val: 645


In [4]:
# 3. Dataset & Transforms
# ==========================================
class SimpleFolderDataset(Dataset):
    def __init__(self, filenames, labels, root_dir, transform=None):
        self.filenames = filenames
        self.labels = labels
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.filenames[idx])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]


train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [5]:
# HyperParameter search function
# ==========================================
def objective(trial):
    print(f"\n🚀 Starting Trial {trial.number} (AlexNet)...")

    # The HyperParameter range
    lr = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])

    print(f"   Settings: Batch={batch_size}, LR={lr:.6f}, Opt={optimizer_name}")

    # Wandb setup
    run = wandb.init(
        project="vangogh-alexnet-optimization",
        name=f"trial_{trial.number}_alexnet",
        config=trial.params,
        reinit=True,
        group="alexnet_search"
    )

    #  DataLoaders
    train_loader = DataLoader(
        SimpleFolderDataset(X_train, y_train, IMAGE_DIR, train_transforms),
        batch_size=batch_size, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        SimpleFolderDataset(X_val, y_val, IMAGE_DIR, val_transforms),
        batch_size=batch_size, shuffle=False, num_workers=0
    )

    model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

    # Features freezing
    for param in model.features.parameters():
        param.requires_grad = False

    # Changing classifier layer
    num_ftrs = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_ftrs, 2)
    # ---------------------------------------------------------

    model = model.to(DEVICE)

    # Optimizer
    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    # Loss weights
    class_weights = torch.tensor([1.0, 5.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)


    best_val_f1 = 0.0

    for epoch in range(NUM_EPOCHS_SEARCH):
        # --- Training epoch ---
        model.train()
        train_loss = 0.0

        loop_train = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS_SEARCH}", leave=False)

        for imgs, lbls in loop_train:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            loop_train.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)

        # --- Validation ---
        model.eval()
        all_preds = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                outputs = model(imgs)

                probs = torch.softmax(outputs, dim=1)[:, 1]
                _, preds = torch.max(outputs, 1)

                all_probs.extend(probs.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.cpu().numpy())

        # Scores calculation
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='binary')
        try:
            val_auc = roc_auc_score(all_labels, all_probs)
        except ValueError:
            val_auc = 0.5

        cm = confusion_matrix(all_labels, all_preds)

        # Trial results
        print(f"📊 Results [Epoch {epoch + 1}/{NUM_EPOCHS_SEARCH}]:")
        print(f"   ⚙️  Params:     LR={lr:.6f} | Batch={batch_size} | Opt={optimizer_name}")
        print(f"   📉 Train Loss: {avg_train_loss:.4f}")
        print(f"   🎯 Val F1:     {val_f1:.4f}")
        print(f"   📈 Val Acc:    {val_acc:.4f}")
        print(f"   💠 Val AUC:    {val_auc:.4f}")
        print(f"   🔲 Conf Mat:   {cm.tolist()}")
        print("-" * 40)

        # Reporting to Wandb
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "val_auc": val_auc,
            "lr": lr,
            "batch_size": batch_size
        })

        trial.report(val_f1, epoch)

        if trial.should_prune():
            print("   ✂️ Trial Pruned.")
            wandb.finish()
            raise optuna.exceptions.TrialPruned()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1

    wandb.finish()
    return best_val_f1

In [6]:
# Main
# ==========================================
if __name__ == "__main__":
    YOUR_WANDB_KEY = "abe064d32b54ea1b7cc4975a901b5f3007609d71"

    print("🔑 Logging into W&B...")
    wandb.login(key=YOUR_WANDB_KEY)

    print("🎯 Creating Optuna Study for AlexNet...")
    study = optuna.create_study(direction="maximize", study_name="AlexNet_Optimization_Weighted")

    print(f"⏳ Optimization started (Timeout: {SEARCH_TIMEOUT_SEC / 60} mins)...")
    try:
        study.optimize(objective, n_trials=None, timeout=SEARCH_TIMEOUT_SEC)
    except KeyboardInterrupt:
        print("Stopping optimization...")

    print("\n✅ Optimization Finished!")
    if len(study.trials) > 0:
        print(f"🏆 Best Val F1: {study.best_value:.4f}")
        print("🏆 Best Params:", study.best_params)

🔑 Logging into W&B...


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\yairshapira\_netrc
wandb: Currently logged in as: guygalanti (guygalanti-tel-aviv-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


🎯 Creating Optuna Study for AlexNet...


[I 2026-01-18 11:27:44,552] A new study created in memory with name: AlexNet_Optimization_Weighted


⏳ Optimization started (Timeout: 30.0 mins)...

🚀 Starting Trial 0 (AlexNet)...
   Settings: Batch=32, LR=0.000774, Opt=SGD


...

📊 Results [Epoch 4/5]:
   ⚙️  Params:     LR=0.000774 | Batch=32 | Opt=SGD
   📉 Train Loss: 0.1778
   🎯 Val F1:     0.7961
   📈 Val Acc:    0.9349
   💠 Val AUC:    0.9591
   🔲 Conf Mat:   [[521, 23], [19, 82]]
----------------------------------------


📊 Results [Epoch 5/5]:
   ⚙️  Params:     LR=0.000774 | Batch=32 | Opt=SGD
   📉 Train Loss: 0.1501
   🎯 Val F1:     0.7719
   📈 Val Acc:    0.9194
   💠 Val AUC:    0.9685
   🔲 Conf Mat:   [[505, 39], [13, 88]]
----------------------------------------


batch_size,▁▁▁▁▁
epoch,▁▃▅▆█
lr,▁▁▁▁▁
train_loss,█▄▂▂▁
val_acc,▁▂█▇▆
val_auc,▁▆▇▆█
val_f1,▁▂█▇▆
batch_size,32
epoch,5
lr,0.00077
train_loss,0.15013


[I 2026-01-18 12:11:54,642] Trial 0 finished with value: 0.8137254901960784 and parameters: {'learning_rate': 0.0007738387156908001, 'batch_size': 32, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.8137254901960784.



✅ Optimization Finished!
🏆 Best Val F1: 0.8137
🏆 Best Params: {'learning_rate': 0.0007738387156908001, 'batch_size': 32, 'optimizer': 'SGD'}
